# Directly downloading fitting sentinel 2 products

In [4]:
import os
import json
import requests
import hashlib
from pathlib import Path
from getpass import getpass
import pandas as pd


In [2]:
product_names =[
 'S2A_MSIL2A_20250512T093141_N0511_R136_T35VMF_20250512T112313.SAFE',
 'S2A_MSIL2A_20250604T094121_N0511_R036_T35VME_20250604T112305.SAFE',
 'S2B_MSIL2A_20230608T093549_N0510_R036_T35VME_20240911T145836.SAFE',
 'S2B_MSIL2A_20230608T093549_N0510_R036_T35VMD_20240911T145836.SAFE',
 'S2A_MSIL2A_20230613T094041_N0510_R036_T35VME_20240927T204801.SAFE',
 'S2A_MSIL2A_20230613T094041_N0510_R036_T35VMD_20240927T204801.SAFE',
 'S2A_MSIL2A_20230620T093041_N0510_R136_T35VMD_20240926T053626.SAFE',
 'S2A_MSIL2A_20230620T093041_N0510_R136_T35VME_20240926T053626.SAFE',
 'S2A_MSIL2A_20230531T093041_N0510_R136_T35VMD_20240908T104259.SAFE',
 'S2A_MSIL2A_20220827T093601_N0510_R036_T35VME_20240717T150803.SAFE',
 'S2A_MSIL2A_20220625T093051_N0510_R136_T35VMF_20240703T013015.SAFE',
 'S2B_MSIL2A_20260523T094029_N0512_R036_T35VME_20260523T115725.SAFE',
 'S2B_MSIL2A_20250521T095029_N0511_R079_T35VMF_20250521T120504.SAFE',
 'S2B_MSIL2A_20250604T093029_N0511_R136_T35VME_20250604T122431.SAFE',
 'S2B_MSIL2A_20250614T093029_N0511_R136_T35VME_20250614T113630.SAFE',
 'S2B_MSIL2A_20220812T093549_N0510_R036_T35VME_20240713T145953.SAFE',
 'S2B_MSIL2A_20220812T093549_N0510_R036_T35VMF_20240713T145953.SAFE',
 'S2B_MSIL2A_20220713T093549_N0510_R036_T35VMF_20240724T031558.SAFE',
 'S2C_MSIL2A_20260717T094031_N0512_R036_T35VME_20260717T130815.SAFE',
 'S2C_MSIL2A_20250615T095051_N0511_R079_T35VMF_20250615T122401.SAFE']

In [3]:
# quiering the selected products

catalogue_url = (
    "https://catalogue.dataspace.copernicus.eu/"
    "odata/v1/Products/OData.CSC.FilterList"
)

payload = {
    "FilterProducts": [
        {"Name": str(name).strip()} #{"Name": name}
        for name in product_names
    ]
}

response = requests.post(
    catalogue_url,
    json=payload,
    timeout=120
)

response.raise_for_status()

catalogue_data = response.json()

catalogue_products = catalogue_data.get(
    "value",
    catalogue_data.get("result", [])
)

print("Products found:", len(catalogue_products))

# for product in catalogue_products:
#     print(
#         product.get("Name"),
#         product.get("Id"),
#         product.get("Online")
#     )

Products found: 20


In [5]:
with open("sentinel2_catalogue.json", "w") as f:
    json.dump(catalogue_products, f, indent=2)

In [6]:
# CONFIGURATION

CATALOGUE_FILE = "sentinel2_catalogue.json"
OUTPUT_DIR = Path("sentinel2_downloads")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOKEN_URL = (
    "https://identity.dataspace.copernicus.eu/"
    "auth/realms/CDSE/protocol/openid-connect/token"
)

DOWNLOAD_URL = (
    "https://download.dataspace.copernicus.eu/"
    "odata/v1/Products({})/$value"
)

In [9]:
# AUTHENTICATION
username = input("Copernicus username: ")
password = getpass("Copernicus password: ")

response = requests.post(
    TOKEN_URL,
    data={
        "client_id": "cdse-public",
        "username": username,
        "password": password,
        "grant_type": "password"
    },
    timeout=60
)

response.raise_for_status()

access_token = response.json()["access_token"]

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {access_token}"
})

print("Authentication successful!")


Copernicus username:  mvalk123@gmail.com
Copernicus password:  ········


Authentication successful!


In [ ]:
# LOAD CATALOGUE

with open(CATALOGUE_FILE, "r") as f:
    products = json.load(f)

print(f"Products in catalogue: {len(products)}")

In [10]:
# --------------------------------------------------
# CHECKSUM VALIDATION
# --------------------------------------------------

def calculate_md5(filename):

    md5 = hashlib.md5()

    with open(filename, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            md5.update(chunk)

    return md5.hexdigest()


def verify_product(filename, product):

    expected = next(
        (
            item["Value"]
            for item in product.get("Checksum", [])
            if item["Algorithm"].upper() == "MD5"
        ),
        None
    )

    if not expected:
        return False

    actual = calculate_md5(filename)

    return actual.lower() == expected.lower()


# --------------------------------------------------
# DOWNLOAD FUNCTION
# --------------------------------------------------

def download_product(product):

    product_id = product["Id"]
    product_name = product["Name"]

    output_file = OUTPUT_DIR / f"{product_name}.zip"
    temp_file = OUTPUT_DIR / f"{product_name}.zip.part"

    print(f"\nProcessing: {product_name}")

    # Check availability
    if not product.get("Online", False):
        print("Product is offline. Skipping.")
        return False

    # Skip already downloaded and verified products
    if output_file.exists():

        if verify_product(output_file, product):
            print("Already downloaded and verified. Skipping.")
            return True

        print("Existing file failed verification. Re-downloading.")
        output_file.unlink()

    # Check whether a partial download exists
    downloaded = temp_file.stat().st_size if temp_file.exists() else 0

    expected_size = product.get("ContentLength")

    if expected_size and downloaded > expected_size:
        temp_file.unlink()
        downloaded = 0

    # Check whether the partial file is actually complete
    if expected_size and downloaded == expected_size:
        if verify_product(temp_file, product):
            os.replace(temp_file, output_file)
            print("Recovered completed download and verified checksum.")
            return True

        temp_file.unlink()
        downloaded = 0

    url = DOWNLOAD_URL.format(product_id)

    headers = {}

    if downloaded > 0:
        headers["Range"] = f"bytes={downloaded}-"
        print(f"Resuming from {downloaded:,} bytes")
    else:
        print("Starting download...")

    try:

        with session.get(
            url,
            headers=headers,
            stream=True,
            timeout=(30, 180),
            allow_redirects=True
        ) as response:

            response.raise_for_status()

            # Append only if the server accepted the Range request.
            if downloaded > 0 and response.status_code == 206:
                mode = "ab"

            else:
                # Server returned the complete file instead of a range.
                mode = "wb"
                downloaded = 0

            with open(temp_file, mode) as f:

                for chunk in response.iter_content(
                    chunk_size=8 * 1024 * 1024
                ):

                    if chunk:
                        f.write(chunk)

        # Validate downloaded size
        if expected_size:
            actual_size = temp_file.stat().st_size

            if actual_size != expected_size:
                print(
                    f"Size mismatch: expected {expected_size:,}, "
                    f"got {actual_size:,} bytes."
                )
                return False

        # Validate checksum
        if not verify_product(temp_file, product):
            print("MD5 checksum verification FAILED!")
            return False

        # Rename only after successful validation
        os.replace(temp_file, output_file)

        print("Download completed and MD5 verified!")
        return True

    except requests.RequestException as e:

        print(f"Download error: {e}")
        print("Partial file retained for a future resume attempt.")
        return False

Products in catalogue: 20


In [ ]:
# MAIN LOOP

successful = 0
failed = []

for i, product in enumerate(products, start=1):

    print(f"\n========== {i}/{len(products)} ==========")

    try:
        if download_product(product):
            successful += 1
        else:
            failed.append(product["Name"])

    except Exception as e:
        print(f"Unexpected error: {e}")
        failed.append(product["Name"])

In [ ]:
# SUMMARY

print("\n========== DOWNLOAD SUMMARY ==========")

print(f"Total products: {len(products)}")
print(f"Successful: {successful}")
print(f"Failed/skipped: {len(failed)}")

if failed:

    print("\nProducts requiring attention:")

    for name in failed:
        print(name)

    with open(OUTPUT_DIR / "failed_downloads.txt", "w") as f:
        f.write("\n".join(failed))

print("\nFinished!")